In [15]:
import h5py
import numpy as np
from scipy.sparse import csc_matrix
from scipy.io import mmwrite

h5_path = "atac_pbmc_10k_nextgem_filtered_peak_bc_matrix.h5"

In [16]:
with h5py.File(h5_path, "r") as f:
    M = f["matrix"]

    data    = M["data"][:].astype(np.float32)
    indices = M["indices"][:]
    indptr  = M["indptr"][:]
    shape   = tuple(M["shape"][:])  # (n_features, n_cells)

    barcodes = M["barcodes"].asstr()[:]

    feats = M["features"]
    intervals = feats["name"].asstr()[:]    
    feat_ids  = feats["id"].asstr()[:]      
    feature_type = feats["feature_type"].asstr()[:]

    X = csc_matrix((data, indices, indptr), shape=shape)

In [17]:
mmwrite(f"atac_pbmc_10k_nextgem_filtered_peak_bc_matrix.matrix.mtx", X)

with open(f"atac_pbmc_10k_nextgem_filtered_peak_bc_matrix.barcodes.tsv", "w") as fh:
    for bc in barcodes:
        fh.write(f"{bc}\n")

def parse_interval(s):
    # "chr1:10000-10100" -> ("chr1", 10000, 10100)
    chrom, coords = s.split(":")
    start, end = coords.split("-")
    return chrom, int(start), int(end)

with open(f"atac_pbmc_10k_nextgem_filtered_peak_bc_matrix.peaks.bed", "w") as fh:
    for itv in intervals:
        chrom, start, end = parse_interval(itv)
        fh.write(f"{chrom}\t{start}\t{end}\n")